# TP 2 · Évaluation & comparaison de configurations RAG
**Module FM02 · IA Générative · M2 Lead Developer Full Stack · ENSITECH**

**Objectif** : concevoir et exécuter un benchmark comparatif de deux configurations RAG, puis recommander la meilleure **chiffres à l'appui**.

| Étape | Consigne de la slide TP 2 | Fichier(s) |
|---|---|---|
| 1 | Construire un jeu de test de **15 questions** avec Ground Truth | `eval/testset.jsonl` |
| 2 | Évaluer **Conf A** (chunks 500, top-k 3) et **Conf B** (chunks 200, top-k 5 + reranking) | `config/conf_a.yaml`, `config/conf_b.yaml`, `src/rag/retriever.py` |
| 3 | Calculer les scores (métriques déterministes + DeepEval) et tracer dans Langfuse | `eval/retrieval_metrics.py`, `eval/llm_judge.py`, `eval/run_benchmark.py` |
| 4 | Rédiger un rapport comparatif | `reports/TP2_rapport_comparatif.md` |

**Livrables** : script d'évaluation, rapport Markdown comparatif chiffré, choix Conf A ou Conf B justifié.


## Étape 0 · Installation (à relancer à chaque nouvelle session Colab)

In [ ]:
# 0.1 GPU disponible ?
!nvidia-smi -L || echo "Pas de GPU : passez en T4 (Exécution > Modifier le type d'exécution) ou utilisez Gemini (0.5)" 

In [ ]:
# 0.2 Récupérer le projet (votre fork GitHub). Alternative : importer le zip puis !unzip
import os
REPO_URL = "https://github.com/VOTRE-COMPTE/fm02-tp-rag.git"   # à remplacer
REPO_DIR = "fm02-tp-rag"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if IN_COLAB:
    %cd {REPO_DIR}
elif os.path.basename(os.getcwd()) == "notebooks":     # Jupyter en local : se placer à la racine du projet
    os.chdir("..")
print(os.getcwd())

In [ ]:
# 0.3 Dépendances (2 à 3 minutes)
!pip install -q -r requirements-eval.txt

In [ ]:
# 0.4 Ollama dans Colab (GPU T4 recommandé) : installation, serveur en tâche de fond, modèle
OLLAMA_MODEL = "qwen3:8b"          # environ 5 Go ; sans GPU : "qwen3:1.7b" ou passer à Gemini
import subprocess, time, urllib.request
if IN_COLAB:
    !sudo apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434"); print("Ollama prêt"); break
    except Exception:
        time.sleep(1)
!ollama pull {OLLAMA_MODEL}

In [ ]:
# 0.5 Clés d'API facultatives, stockées dans les secrets Colab (icône clé à gauche) :
#   GEMINI_API_KEY (secours si pas de GPU, clé gratuite sur aistudio.google.com)
#   LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL (traces, compte gratuit sur cloud.langfuse.com)
# En local : fichier .env (voir .env.example). Ne jamais écrire une clé en clair dans le notebook.
import os
try:
    from google.colab import userdata
    for k in ["GEMINI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
print({k: ("ok" if os.environ.get(k) else "absente") for k in ["GEMINI_API_KEY", "LANGFUSE_PUBLIC_KEY"]})

In [ ]:
# 0.6 Rendre le code de src/ importable, et le recharger automatiquement après chaque modification
import sys
sys.path[:0] = ["src", "."]
%load_ext autoreload
%autoreload 2

## Étape 1 · Jeu de test (Ground Truth)
Chaque ligne de `eval/testset.jsonl` contient : `id`, `question`, `ground_truth` (réponse de référence), `expected_sources` (fichiers attendus), `expected_keywords` (mots-clés vérifiables, variantes séparées par `|`) et `type`.
Un bon jeu de test **couvre les cas difficiles** : tableau, PDF, HTML, identifiant exact (RH-07), question multi-documents, question hors corpus.

### TODO · Compléter le jeu de test à 15 questions
Le fichier fourni contient 5 exemples. Ajoutez 10 questions en respectant la répartition suivante : au moins 2 questions sur un tableau, 1 sur le PDF, 1 sur le HTML, 1 avec un identifiant exact, 1 multi-documents, 1 hors corpus. Vérifiez chaque réponse **dans le document source** : une erreur dans le Ground Truth fausse tout le benchmark.

In [ ]:
import json, collections
testset = [json.loads(l) for l in open("eval/testset.jsonl", encoding="utf-8") if l.strip()]
print(len(testset), "questions ·", dict(collections.Counter(c["type"] for c in testset)))
for c in testset[:3]:
    print("-", c["id"], c["question"], "->", c["expected_sources"])

## Étape 2 · Exécuter Conf A et Conf B (métriques déterministes)
On partage **le même modèle d'embedding et le même LLM** entre les deux configurations : seule la récupération change, c'est ce qu'on veut mesurer. Chaque configuration a sa propre collection ChromaDB.

In [ ]:
from rag.config import load_config
from rag.embeddings import Embedder
from rag.llm import get_llm
from rag.pipeline import RAGPipeline
from eval.run_benchmark import evaluate_config, summarize

cfg_a, cfg_b = load_config("config/conf_a.yaml"), load_config("config/conf_b.yaml")
embedder = Embedder(cfg_a.embedding_model)
llm = get_llm(cfg_a)
pipes = {c.name: RAGPipeline(c, embedder=embedder, llm=llm) for c in (cfg_a, cfg_b)}
for name, p in pipes.items():
    print(name, p.index("data/corpus", reset=True))

In [ ]:
results = {name: evaluate_config(p, testset) for name, p in pipes.items()}

In [ ]:
import pandas as pd
summary = pd.DataFrame([summarize(rows) for rows in results.values()]).set_index("config")
summary

> **Pour aller plus loin** : lancez aussi `config/conf_c.yaml` (chunking par sections + recherche hybride BM25 + reranking). Gagne-t-elle sur les questions à identifiant exact (`RH-07`, `SEC-2`, `P2`) ?

## Étape 3 · LLM-as-a-judge (DeepEval) et traces Langfuse
Les 4 métriques DeepEval correspondent à la **RAG Triad** du cours. Le juge est Gemini si une clé existe (modèle différent du générateur), sinon le même modèle Ollama.

Durée indicative : 15 questions × 2 configurations × 4 métriques, soit environ 120 évaluations (10 à 25 min selon le juge). Commencez avec `LIMIT = 3`.

In [ ]:
from eval.llm_judge import build_metrics, get_judge, judge_case

LIMIT = 3            # passer à None pour tout le jeu de test
PAUSE_S = 4          # pause entre appels au juge (quota gratuit Gemini)
judge = get_judge("auto", ollama_model=cfg_a.ollama_model)
metrics = build_metrics(judge)
print("juge :", judge.get_model_name())

for name, rows in results.items():
    for row, case in zip(rows[:LIMIT], testset[:LIMIT]):
        row.update(judge_case(metrics, case["question"], row["answer"], row["_contexts"], case["ground_truth"], pause_s=PAUSE_S))
        print(name, case["id"], {m: row.get(m) for m in metrics})

In [ ]:
summary = pd.DataFrame([summarize(rows) for rows in results.values()]).set_index("config")
summary

**Traces Langfuse** : si les clés sont définies (étape 0.5), chaque question a produit une trace `rag-ask` avec ses étapes `retrieval` et `generation`, étiquetée `conf_a` ou `conf_b`. Dans l'interface Langfuse, filtrez par étiquette et comparez la latence de la recherche (avec et sans reranking) et le nombre de tokens envoyés au LLM.

> **Pour aller plus loin** : refaites l'évaluation avec un autre juge (Ollama contre Gemini) ou avec **RAGAS** sur les mêmes réponses. Les classements A/B sont-ils stables quand le juge change ?

## Étape 4 · Analyse et rapport
Le script complet en ligne de commande produit les mêmes résultats dans `eval/results/` :
```bash
python eval/run_benchmark.py --configs config/conf_a.yaml config/conf_b.yaml --judge auto --pause 4
```

In [ ]:
import matplotlib.pyplot as plt
cols = [c for c in ["hit@k", "mrr", "answer_correct", "faithfulness", "answer_relevancy", "contextual_precision", "contextual_recall"] if c in summary]
summary[cols].T.plot.bar(figsize=(10, 4), ylim=(0, 1.05), rot=30, title="Conf A vs Conf B")
plt.tight_layout(); plt.show()
summary[["latency_p50_s", "retrieval_p50_s", "prompt_tokens_mean"]]

In [ ]:
# Analyse des écarts : sur quelles questions les deux configurations divergent-elles ?
details = pd.concat([pd.DataFrame(r).drop(columns="_contexts") for r in results.values()])
pivot = details.pivot_table(index=["id", "type"], columns="config", values=["answer_correct", "hit@k"])
pivot[pivot[("answer_correct", "conf_a")] != pivot[("answer_correct", "conf_b")]]

In [ ]:
# Par type de question
details.groupby(["type", "config"])["answer_correct"].mean().unstack()

In [ ]:
from pathlib import Path
from eval.run_benchmark import markdown_table, write_csv

Path("eval/results").mkdir(parents=True, exist_ok=True)
for name, rows in results.items():
    write_csv(rows, Path(f"eval/results/details_{name}.csv"))
table = markdown_table([summarize(rows) for rows in results.values()])
Path("eval/results/summary.md").write_text(table, encoding="utf-8")
print(table)

### TODO · Rédiger le rapport
Complétez `reports/TP2_rapport_comparatif.md` : tableau des résultats, analyse d'au moins 3 questions où A et B divergent (cause probable : chunk coupé, mauvais classement, réponse non fidèle...), coût (latence, tokens), recommandation argumentée et limites de l'étude.

> **Pour aller plus loin** : (1) répétez chaque configuration 3 fois pour mesurer la variance ; (2) faites varier un seul paramètre (top-k = 3, 5, 8) pour isoler son effet ; (3) ajoutez la configuration C au rapport.

**Rendu** : commit et push du code, de `eval/testset.jsonl`, de `eval/results/` et du rapport.